# Run Hyperparameter Comparison

Comparing differences between hyperparameters as read from the saved args.yaml files

In [1]:
# NOTE: run only once!
import os
os.chdir('..')

In [2]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

In [ ]:
# Define run names to read
run_names = [
    ("long_runs",   "150ep_scale_trans_off"),
    ("optimizers",  "opt_150ep_SGD_mom_lower"),
    ("long_runs",   "150ep_bch_32"),
    ("long_runs",   "250ep"),
    ("optimizers",  "opt_150ep_SGD_lr0_lower"),
    ("long_runs",   "1st_150ep"),
    ("long_runs",   "150ep_crop"),
    ("long_runs",   "150ep_lr0_half"),
    ("optimizers",  "opt_150ep_SGD_mom_higher"),
    ("long_runs",   "180ep_normal"),
    ("optimizers",  "opt_150ep_SGD_lr0"),
    ("optimizers",  "opt_150ep_SGD"),
    ("long_runs",   "150ep_hsv"),
    ("optimizers",  "opt_150ep_adamw_lr0"),
    ("long_runs",   "150ep_flipup"),
    ("long_runs",   "150ep_no_drone"),
    ("optimizers",  "opt_150ep_adamw"),
    ("long_runs",   "150ep_imgsz_480"),
    ("long_runs",   "150ep_bch_8"),
    ("long_runs",   "90ep"),
    ("long_runs",   "180ep_degrees"),
    ("long_runs",   "150ep_imgsz_400"),
    ("long_runs",   "150ep_degrees"),
    ("long_runs",   "150ep_normal"),
]

In [61]:
ignore_names = ["250ep", "90ep", "150ep_no_drone", "1st_150ep", "opt_150ep_RMSProp"]
run_names2 = []
for item in Path("./runs_puhti").glob("*"):
    if item.is_dir():
        run_group = item.name
        for subdir in item.rglob("*/best.pt"):
            run_name = subdir.parent.parent.name
            if run_name not in ignore_names:
                # print(run_group, run_name)
                run_names2.append((run_group, run_name))

In [ ]:
# 
args_dicts = []
for run_group, run_name in run_names2:

    args_file = Path("runs_puhti") / run_group / run_name / "args.yaml"
    if not args_file.exists():
        raise Exception(f"args.yaml file doesn't exist: \"{args_file.resolve()}\"")

    with open(str(args_file), 'r') as f:
        data = yaml.safe_load(f)
    data = data | {"name": run_name, "project": run_group}
    args_dicts.append(data)

In [74]:
# create df
df = pd.DataFrame(args_dicts)
df = df.drop(["cfg", "save_dir"], axis=1)
cols = list(df.columns)
priority = ["name", "project", "data"]
cols = priority + sorted(c for c in cols if c not in priority)
df = df[cols]
df

,name,project,data,agnostic_nms,amp,augment,auto_augment,batch,bgr,box,...,val,verbose,vid_stride,visualize,warmup_bias_lr,warmup_epochs,warmup_momentum,weight_decay,workers,workspace
0,150ep_bch_32,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,32,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
1,150ep_bch_8,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,8,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
2,150ep_crop,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
3,150ep_degrees,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
4,150ep_flipup,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
5,150ep_hsv,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
6,150ep_imgsz_320,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
7,150ep_imgsz_400,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
8,150ep_imgsz_480,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None
9,150ep_lr0_half,long_runs,YAML/RDD2022_PP_kaikki.yaml,False,True,False,randaugment,16,0.0,7.5,...,True,True,1,False,0.1,3.0,0.8,0.0005,10,None


In [76]:
# 
cols_to_keep = [
    col for col in df.columns
    if len(set(df[col])) > 1
]
df = df[cols_to_keep]
df.head()
df.to_csv('analytics/hyperparameter_comparing.csv', index=False)